In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'paths.py').exists())))
import paths as P
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tangermeme.plot import plot_logo
from tangermeme.io import read_meme
from tangermeme.utils import one_hot_encode, characters
import pyBigWig
from scipy.stats import pearsonr, spearmanr, levene, brunnermunzel
from scipy import stats

from tqdm import tqdm
import seaborn as sns


## 1. Loading

In [ ]:
id = 2
epi = 'H3K27ac'

### 1.1 Loading perturb scores and the corresponding sequences.

In [ ]:
seq = np.load(str(P.HUMAN_INTER_DIR / 'GM12878/seq_9.npy'))
print(seq.shape)

In [ ]:
attr_files = {
    'GM12878' : f'{P.HUMAN_INTER_DIR}/GM12878/{epi}/attr_9_filled.npy', 
    'H1ESC'    : f'{P.HUMAN_INTER_DIR}/H1ESC/{epi}/attr_9_filled.npy',
    'K562'    : f'{P.HUMAN_INTER_DIR}/K562/{epi}/attr_9_filled.npy',
}

In [ ]:
def standardize(v):
    score = np.sum(v, axis=0)
    std = np.std(score)
    print(std)
    normalized = v / std
    return normalized

def quantile(v, l=20, q=0.999):
    score = np.sum(v, axis=0)
    c = np.cumsum(score)
    s = (c[l:]-c[:-l])/l
    return np.quantile(np.abs(s), q)

attrs_raw = {}
attrs_standardized = {}
thres = {}
for k, v in attr_files.items():
    attrs_raw[k] = np.load(v)[0]
    print(k)
    print(attrs_raw[k].shape)
    print(np.max(attrs_raw[k]))
    print(np.min(attrs_raw[k]))
    attrs_standardized[k] = standardize(attrs_raw[k])
    print(np.max(attrs_standardized[k]))
    print(np.min(attrs_standardized[k]))
    thres[k] = quantile(attrs_standardized[k])
    print(thres[k])



### 1.2 Loading motif data

In [ ]:
all_hits = pd.read_csv(str(P.MOTIF_MATCH_CHR9_BED), sep='\t', header=None)
# all_hits.columns = ['chromosome', 'start', 'end', 'archetype_name', 'motif_score', 'strand', 'motif_name', 'seq']
all_hits.columns = ['chromosome', 'start', 'end', 'motif_name', 'motif_score', 'strand', 'seq']

In [ ]:
cols = list(all_hits.columns)
cols = cols[3:4]+cols[:3]+cols[4:]
all_hits = all_hits[cols]


In [ ]:
all_hits

In [ ]:
jaspar_hits = all_hits[all_hits['motif_name'].str.contains('_MA')]
jaspar_hits = jaspar_hits[~jaspar_hits['motif_name'].str.contains('_MADS')]

In [ ]:
jaspar_hits

In [ ]:
jaspar_human_hits = jaspar_hits[
    jaspar_hits['motif_name'].str.split('-', n=1).str[0].str.isupper()
]


### 1.4 loading predictions

In [ ]:
pred_files = {
    'GM12878' : f'{P.EPI_EVO2HIC_DIR}/GM12878/9.npy', 
    'H1ESC'    : f'{P.EPI_EVO2HIC_DIR}/H1ESC/9.npy',
    'K562'    : f'{P.EPI_EVO2HIC_DIR}/K562/9.npy',
}

In [ ]:
preds = {}
for k, v in pred_files.items():
    preds[k] = np.load(v)[id]
    print(k)
    print(preds[k].shape)

## 2. Sites finding 

### 2.1 check motif distribution

In [ ]:
motif_hits = jaspar_human_hits[jaspar_human_hits['motif_name'] == 'IRF2_MA0051.1']
# motif_hits = motif_hits[motif_hits['motif_score']>11]
score_H1ESC = []
score_GM12878 = []
score_0 = []
l = None
# fig, axes = plt.subplots(
#     6, 1,
#     figsize=(6, 5),
#     gridspec_kw={'hspace': 1}
# )
attr_H1ESC = []
attr_GM12878 = []
cnt = 0
thres = 5
for row in motif_hits.itertuples():
    s, e = int(row.start), int(row.end)
    score_0.append(row.motif_score)

    if l is not None:
        assert e-s == l
    else:
        l = e-s

    cH1ESC = np.abs(attrs_standardized['H1ESC'][:, s:e].sum(axis=0).mean())
    attr_H1ESC.append(cH1ESC)
    if cH1ESC > thres:
        score_H1ESC.append(row.motif_score)
    cGM12878 = np.abs(attrs_standardized['GM12878'][:, s:e].sum(axis=0).mean())
    attr_GM12878.append(cGM12878)
    if cGM12878 > thres:
        score_GM12878.append(row.motif_score)

    # if cGM12878 > 10 and cH1ESC<1 and preds['H1ESC'][s//2000]<0.2: 
    #     axes[cnt].set_title(f'chr9:{s-20}-{e+20}')
    #     plot_logo(attrs_standardized['GM12878'], axes[cnt], annotations=None, start=s-10, end=e+10, score_key='motif_score', show_extra=False)
    #     axes[cnt].set_xticks([])
    #     cnt += 1
    #     plot_logo(attrs_standardized['H1ESC'], axes[cnt], annotations=None, start=s-10, end=e+10, score_key='motif_score', show_extra=False)
    #     axes[cnt].set_xticks([])
    #     cnt += 1
    #     # y0min, y0max = axes[cnt-1].get_ylim()
    #     # y1min, y1max = axes[cnt-2].get_ylim()
    #     # ymin, ymax = min(y0min, y1min), max(y0max, y1max)
    #     # print(ymin, ymax)
    #     # axes[cnt-1].set_ylim(ymin, ymax)
    #     # axes[cnt-2].set_ylim(ymin, ymax)

    #     L, R = s//2000-5, e//2000+6
    #     print(s//2000*2)
    #     axes[cnt].set_title(f'chr9:{L*2}kb-{R*2}kb')
    #     axes[cnt].plot(range(L*2, R*2, 2), preds['GM12878'][L:R], linewidth=0.3)
    #     axes[cnt].fill_between(
    #         x=range(L*2, R*2, 2), 
    #         y1=preds['GM12878'][L:R],
    #         alpha=0.5
    #     )
    #     cnt += 1
    #     axes[cnt].plot(range(L*2, R*2, 2), preds['H1ESC'][L:R], linewidth=0.3)
    #     axes[cnt].fill_between(
    #         x=range(L*2, R*2, 2), 
    #         y1=preds['H1ESC'][L:R],
    #         alpha=0.5
    #     )
    #     cnt += 1
    #     y0min, y0max = axes[cnt-1].get_ylim()
    #     y1min, y1max = axes[cnt-2].get_ylim()
    #     ymin, ymax = min(y0min, y1min), max(y0max, y1max)
    #     print(ymin, ymax)
    #     axes[cnt-1].set_ylim(ymin, ymax)
    #     axes[cnt-2].set_ylim(ymin, ymax)

    #     if cnt>=4:
    #         break

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# density curves
sns.kdeplot(score_0, fill=True, bw_adjust=1.0, label='all motif sites')
sns.kdeplot(score_GM12878, fill=True, bw_adjust=1.0, label='high attribution sites in GM12878')
# sns.kdeplot(score_H1ESC, fill=True, bw_adjust=1.0, label='GM12878 high attribution sites')

plt.xlabel("Tomtom score")
plt.ylabel("Density")
plt.tight_layout()
plt.legend()
plt.show()


### 2.1 motif enrichment for high attribution

In [ ]:
from scipy.stats import fisher_exact
def tail_prob_test(x0, x1, thres, option='fixed', alternative='greater'):
    if option == 'fixed':
        k0 = np.sum(x0 > thres)
        k1 = np.sum(x1 > thres)
    elif option == 'percentage':
        thres = np.quantile(x1, thres)
        k0 = np.sum(x0 > thres)
        k1 = np.sum(x1 > thres)

    n0, n1 = len(x0), len(x1)
    table = np.array([[k0, n0-k0], [k1, n1-k1]])
    stat, p = fisher_exact(table, alternative=alternative)
    # p_pool = (k0 + k1) / (n0 + n1)
    # se = np.sqrt(p_pool * (1 - p_pool) * (1/n0 + 1/n1))
    # z = (k0/n0 - k1/n1) / se
    # p = 1 - stats.norm.cdf(z) if alternative == 'greater' else stats.norm.cdf(z)
    return {
        "threshold": thres,
        "k0": k0,
        "n0": n0,
        "k1": k1,
        "n1": n1,
        "stat": stat,
        "p": p,
    }

In [ ]:
def get_scores(score, motif_hits, local_background_size=10):
    score = score.sum(axis=0)
    L = score.shape[-1]
    l = None
    contribs, local_backgrounds = [], []
    for row in motif_hits.itertuples():
        s, e = int(row.start), int(row.end)
        if s < 0 or e > L or e <= s:
            continue

        if l is not None:
            assert e-s == l
        else:
            l = e-s

        contrib = score[s:e].mean()
        contribs.append(contrib)

        s = np.cumsum(score[s-local_background_size*l:e+local_background_size*l])
        local_background = (s[:-l] - s[l:])/l

        local_backgrounds.append(local_background)

    contribs = np.array(contribs)
    local_backgrounds = np.concatenate(local_backgrounds)
    return np.abs(contribs), np.abs(local_backgrounds)

In [ ]:
def calc_motif_statistics(
        motif_name, 
        cells = None,
        local_background_size = 10, 
    ):
    motif_hits = all_hits[all_hits['motif_name'] == motif_name]
    if len(motif_hits) == 0:
        for motif in all_hits['motif_name'].unique():
            if motif.startswith(motif_name):
                print(f'You may look at {motif}')
        return

    if cells is None:
        cells = attrs_standardized.keys()

    statistics = {}

    for k in cells:
        contribs, local_backgrounds = get_scores(attrs_standardized[k], motif_hits, local_background_size)
        statistics[k] = tail_prob_test(contribs, local_backgrounds, thres[k], alternative='greater')
        # statistics[k] = tail_prob_test(contribs, local_backgrounds, 0.998, option='percentage', alternative='greater')

    return statistics


In [ ]:
calc_motif_statistics('MSC_MA0665.1')

In [ ]:
all_motifs = sorted(jaspar_human_hits['motif_name'].unique())
all_motifs += ['RARG_nuclearreceptor_6']
all_motifs

In [ ]:
stat = {}
for motif in all_motifs:
    stat[motif] = calc_motif_statistics(motif)

rows = []
for motif, cell_data in stat.items():
    for cell, stat in cell_data.items():
        row = {'motif_name': motif, 'cell_line': cell}
        row.update(stat)
        rows.append(row)

df = pd.DataFrame(rows)

# === save to disk ===
df.to_csv(f"{P.RESULT("main/fig5")}/motif_enrichment_stats_{epi}_positive.csv", index=False, sep='\t')


In [ ]:
import pandas as pd

# read
df = pd.read_csv(f"{P.RESULT("main/fig5")}/motif_enrichment_stats_{epi}_positive.csv", sep="\t")
df['k1']/df['n1']

In [ ]:
import pandas as pd

# read
df = pd.read_csv(f"{P.RESULT("main/fig5")}/motif_enrichment_stats_{epi}_positive.csv", sep="\t")

# pivot: p-values per motif x cell line
P = df.pivot(index="motif_name", columns="cell_line", values="p")

# threshold
alpha_sig = 5e-2
alpha_ns  = 5e-2

specific_hits = {}
for cell in P.columns:
    cond_sig_here  = P[cell] < alpha_sig
    cond_ns_others = P.drop(columns=[cell]).gt(P[cell]).fillna(False).all(axis=1)
    mask = cond_sig_here & cond_ns_others

    result = P.loc[mask].sort_values(by=cell)
    specific_hits[cell] = result

    print(f"{cell}: {len(result)} motifs")

specific_hits


In [ ]:
import pandas as pd

# read
df = pd.read_csv(f"{P.RESULT("main/fig5")}/motif_enrichment_stats_{epi}_positive.csv", sep="\t")

# pick the z-statistic column (adapts to the table header)
z_cols_try = ["zstat", "z_stat", "zscore", "z"]
z_col = next((c for c in z_cols_try if c in df.columns), None)
if z_col is None:
    raise ValueError(f"no z-statistic column found; rename one column to one of: {z_cols_try}")

# pivot: z-values per motif x cell line
Z = df.pivot(index="motif_name", columns="cell_line", values=z_col)

dz = 10.0  # threshold for 'significantly larger' (adjustable)

specific_hits = {}
for cell in Z.columns:
    # z of this cell
    z_here = Z[cell]

    # maximum z over the other cells (per row)
    z_others_max = Z.drop(columns=[cell]).max(axis=1)

    # difference to the runner-up (maximum over the other cells)
    delta = z_here - z_others_max

    # keep motifs at least dz above every other cell
    mask = delta >= dz

    # result: z of every cell plus a delta column
    result = Z.loc[mask].copy()
    result["delta_vs_next_best"] = delta[mask]

    # sort by this cell's z, descending
    result = result.sort_values(by=cell, ascending=False)

    specific_hits[cell] = result

    print(f"{cell}: {len(result)} motifs")

specific_hits
